Tutorial stuff to get PyPSA working in a single-node model, building towards the PyPSA-RSA style model

### Import cell

In [ ]:
import logging
import re
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import pypsa

from _helpers import mock_snakemake
# Updated imports for PyPSA 0.34.1 - using linopy-based optimization
from pypsa.descriptors import get_switchable_as_dense as get_as_dense, expand_series

# Importing necessary functions from the custom modules
from _helpers import configure_logging, remove_leap_day, normalize_and_rename_df, assign_segmented_df_to_network, load_scenario_definition, add_missing_carriers
from add_electricity import load_extendable_parameters#, update_transmission_costs
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # Comment out for debugging and development
from custom_constraints import set_operational_limits, ccgt_steam_constraints, reserve_margin_constraints, annual_co2_constraints
from custom_constraints import add_national_capacity_constraints

idx = pd.IndexSlice

In [ ]:
from pypsa_rsa.scripts import add_electricity

In [ ]:

snakemake = mock_snakemake(
    'prepare_and_solve_network', 
    **{
        'scenario':'TEST',
    }
)

### Example runs

In [ ]:
file_path = Path("~/showcase/pypsa-rsa/networks/TEST/elec/capacity-TEST.nc").expanduser()
# Load the network from the NetCDF file
n = pypsa.Network(file_path)
add_missing_carriers(n)
scenario_setup = load_scenario_definition(snakemake)

In [ ]:
# Use fewer time steps for testing
n_small= n.copy()
n_small.snapshots = n.snapshots[:168]  # Just 24 hours

In [ ]:
solve_network(n_small, n_small.snapshots)

In [ ]:
n_small.model

### Explore results

In [ ]:
n_small.snapshot_weightings